## CELDA 1 — Preparación del panel para el HBM (PM2.5) y estructura espacial \(W\)

**Objetivo:** dejar listo el dataset que va a consumir el Modelo Bayesiano Jerárquico (HBM) para PM2.5 en formato **celda–mes**, con:
- variable respuesta \(y_{i,t}\),
- covariables estandarizadas \(X_{i,t}\) (incluyendo proxies A–D),
- índices internos para efectos espaciales y temporales,
- y la vecindad espacial \(W\) (Queen) en forma de aristas \((i \rightarrow j)\).

### ¿Qué hace esta celda?

1. **Detecta la raíz del proyecto automáticamente (VS Code):**
   - Si el notebook se ejecuta desde la carpeta `CODIGO/`, usa el directorio padre como raíz del proyecto.
   - Con esa raíz, arma las rutas reales a:
     - `BASES DE DATOS/PANEL AMBIENTAL INTEGRADO/grid_3km_AD_mensual_2020_2024.csv`
     - `BASES DE DATOS/PANEL AMBIENTAL INTEGRADO/W_grid_3km_queen.csv`

2. **Carga los datos del panel A–D en grilla y la vecindad \(W\):**
   - Panel base: concentraciones IDW + viento + proxies A–D por celda–mes.
   - Vecindad: lista de pares `cell_id → neighbor_id` (Queen contiguity).

3. **Estandariza la variable temporal:**
   - Convierte la columna `fecha` a formato mensual `YYYY-MM` para definir el índice temporal \(t\).

4. **Define la variable respuesta del HBM para PM2.5:**
   - Usa `PM25_idw` como respuesta.
   - (Opcional) aplica transformación logarítmica: \(y_{i,t}=\log(PM25\_idw+\epsilon)\) para estabilizar varianza y garantizar positividad.

5. **Selecciona las covariables \(X_{i,t}\):**
   - Incluye como base:
     - `vel_viento_idw`
     - `diff_PM25_grid`, `grad_PM25_grid`, `adv_proxy_PM25_grid`
   - (Opcional) permite agregar meteorología si ya está integrada y declaras los nombres exactos.

6. **Construye índices internos para el HBM:**
   - `cell_idx`: índice entero para cada `cell_id` (efecto espacial).
   - `time_idx`: índice entero para cada mes `fecha` (efecto temporal).

7. **Estandariza las covariables (z-score):**
   - Convierte cada covariable a escala comparable:
     \[
     X^*=\frac{X-\mu}{\sigma}
     \]
   - Si alguna covariable es constante o inválida, se fija en 0 (no aporta señal al modelo).

8. **Reexpresa la vecindad \(W\) en índices internos:**
   - Convierte `cell_id` y `neighbor_id` a `i` y `j` usando el mapeo `cell_map`.
   - Esto deja lista la estructura espacial para el efecto CAR/ICAR.

### Salidas (archivos que guarda para la CELDA 2)

- `CODIGO/HBM_OUT/HBM_panel_PM25.parquet`  
  Panel final con columnas:
  - `cell_id`, `fecha`, `cell_idx`, `time_idx`, `y`, y covariables estandarizadas.

- `CODIGO/HBM_OUT/HBM_W_edges_queen.csv`  
  Aristas de vecindad en índices:
  - `i`, `j`

- `CODIGO/HBM_OUT/HBM_meta.json`  
  Metadatos del ajuste:
  - contaminante, si se usó log, covariables incluidas, número de celdas y meses.

**Resultado esperado al final de la celda:**  
tener el panel HBM y la estructura \(W\) listos para ajustar el HBM en la **CELDA 2** (modelo base para PM2.5).

In [1]:
# ✅ CELDA 1 (actualizada a tus rutas en VS Code)
# Prepara panel HBM (PM2.5) + índices + covariables estandarizadas + W (Queen)

import pandas as pd
import numpy as np
import json
from pathlib import Path

# =========================
# 0) Detectar raíz del proyecto (asumiendo que ejecutas desde /CODIGO)
# =========================
cwd = Path.cwd()
if cwd.name.lower() == "codigo":
    PROJECT_ROOT = cwd.parent
else:
    # si no estás en CODIGO, busca hacia arriba una carpeta que contenga "BASES DE DATOS"
    PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "BASES DE DATOS").exists()), cwd)

DATA_DIR = PROJECT_ROOT / "BASES DE DATOS" / "PANEL AMBIENTAL INTEGRADO"

DATA_CSV = DATA_DIR / "grid_3km_AD_mensual_2020_2024.csv"
W_CSV    = DATA_DIR / "W_grid_3km_queen.csv"

OUT_DIR  = PROJECT_ROOT / "CODIGO" / "HBM_OUT"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("📌 PROJECT_ROOT =", PROJECT_ROOT)
print("📌 DATA_CSV     =", DATA_CSV, "| exists:", DATA_CSV.exists())
print("📌 W_CSV        =", W_CSV,   "| exists:", W_CSV.exists())
print("📌 OUT_DIR      =", OUT_DIR)

if not DATA_CSV.exists():
    raise FileNotFoundError(f"No encuentro el archivo: {DATA_CSV}")
if not W_CSV.exists():
    raise FileNotFoundError(f"No encuentro el archivo: {W_CSV}")

# =========================
# 1) Configuración (PM2.5 primero)
# =========================
POLL = "PM25"       # luego repetimos con "NO2" y "O3"
USE_LOG = True
EPS = 1e-6

X_BASE = [
    "vel_viento_idw",
    f"diff_{POLL}_grid",
    f"grad_{POLL}_grid",
    f"adv_proxy_{POLL}_grid",
]

# Si tu panel ya tiene meteo integrada, pon aquí los nombres exactos (si no, déjalo vacío)
METEO_EXACT = []   # ejemplo: ["T2M", "RH2M", "PRECTOTCORR"]

# =========================
# 2) Cargar datos
# =========================
df = pd.read_csv(DATA_CSV)
w  = pd.read_csv(W_CSV)

# =========================
# 3) Normalizar columna de fecha (acepta 'fecha' o 'Fecha')
# =========================
if "fecha" not in df.columns:
    if "Fecha" in df.columns:
        df = df.rename(columns={"Fecha": "fecha"})
    else:
        raise ValueError("No encuentro columna 'fecha' (ni 'Fecha') en el dataset.")

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
if df["fecha"].isna().any():
    raise ValueError("No pude convertir 'fecha' a datetime. Revisa el formato de esa columna.")

df["fecha"] = df["fecha"].dt.to_period("M").astype(str)  # "YYYY-MM"

# =========================
# 4) Definir respuesta y (PM25_idw)
# =========================
y_col = f"{POLL}_idw"
if y_col not in df.columns:
    raise ValueError(f"No existe la columna {y_col} en el CSV.")

if USE_LOG:
    df["y"] = np.log(pd.to_numeric(df[y_col], errors="coerce").astype(float) + EPS)
else:
    df["y"] = pd.to_numeric(df[y_col], errors="coerce").astype(float)

# =========================
# 5) Seleccionar covariables X disponibles
# =========================
missing_base = [c for c in X_BASE if c not in df.columns]
if missing_base:
    print("⚠️ Faltan covariables base esperadas:", missing_base)

X_cols = [c for c in X_BASE if c in df.columns] + [c for c in METEO_EXACT if c in df.columns]
if len(X_cols) == 0:
    raise ValueError("No encontré covariables X. Revisa nombres de columnas del dataset.")

# =========================
# 6) Índices: celda y tiempo
# =========================
cell_ids = np.sort(df["cell_id"].unique())
time_ids = np.sort(df["fecha"].unique())

cell_map = {cid: i for i, cid in enumerate(cell_ids)}
time_map = {tt:  j for j, tt  in enumerate(time_ids)}

df["cell_idx"] = df["cell_id"].map(cell_map).astype(int)
df["time_idx"] = df["fecha"].map(time_map).astype(int)

# =========================
# 7) Estandarizar X (z-score)
# =========================
X_z = pd.DataFrame(index=df.index)
for c in X_cols:
    v = pd.to_numeric(df[c], errors="coerce")
    mu = np.nanmean(v)
    sd = np.nanstd(v)
    if sd == 0 or np.isnan(sd):
        X_z[c] = 0.0
        print(f"⚠️ Covariable constante/ inválida -> {c} (se deja en 0).")
    else:
        X_z[c] = (v - mu) / sd

# =========================
# 8) Preparar W (edges) en índices internos
# =========================
if not set(["cell_id", "neighbor_id"]).issubset(w.columns):
    raise ValueError("W_grid_3km_queen.csv debe tener columnas: 'cell_id' y 'neighbor_id'.")

w = w[w["cell_id"].isin(cell_ids) & w["neighbor_id"].isin(cell_ids)].copy()
w["i"] = w["cell_id"].map(cell_map).astype(int)
w["j"] = w["neighbor_id"].map(cell_map).astype(int)

# =========================
# 9) Guardar salidas para la CELDA 2
# =========================
panel_out = df[["cell_id","fecha","cell_idx","time_idx","y"]].join(X_z)
edges_out = w[["i","j"]].drop_duplicates()

panel_path = OUT_DIR / f"HBM_panel_{POLL}.parquet"
edges_path = OUT_DIR / "HBM_W_edges_queen.csv"
meta_path  = OUT_DIR / "HBM_meta.json"

panel_out.to_parquet(panel_path, index=False)
edges_out.to_csv(edges_path, index=False)

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(
        {"POLL": POLL, "USE_LOG": USE_LOG, "X_cols": X_cols, "n_cells": len(cell_ids), "n_months": len(time_ids)},
        f, ensure_ascii=False, indent=2
    )

print("\n===== RESUMEN =====")
print("POLL:", POLL, "| filas:", len(panel_out), "| celdas:", len(cell_ids), "| meses:", len(time_ids))
print("Y:", ("log("+y_col+")" if USE_LOG else y_col))
print("X:", X_cols)
print("Edges W:", len(edges_out))
print("\n✅ Guardado:")
print(" -", panel_path)
print(" -", edges_path)
print(" -", meta_path)

📌 PROJECT_ROOT = d:\TRABAJO DE GRADO BEN-MAP
📌 DATA_CSV     = d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\PANEL AMBIENTAL INTEGRADO\grid_3km_AD_mensual_2020_2024.csv | exists: True
📌 W_CSV        = d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\PANEL AMBIENTAL INTEGRADO\W_grid_3km_queen.csv | exists: True
📌 OUT_DIR      = d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT
⚠️ Faltan covariables base esperadas: ['vel_viento_idw']

===== RESUMEN =====
POLL: PM25 | filas: 15240 | celdas: 254 | meses: 60
Y: log(PM25_idw)
X: ['diff_PM25_grid', 'grad_PM25_grid', 'adv_proxy_PM25_grid']
Edges W: 1610

✅ Guardado:
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_panel_PM25.parquet
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_W_edges_queen.csv
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_meta.json


## CELDA 2 — Ajuste del HBM base (PM2.5) con estructura espacio–temporal y salida de superficies finales

**Objetivo:** ajustar el Modelo Bayesiano Jerárquico (HBM) para PM2.5 sobre el panel **celda–mes**, incorporando:
- efectos fijos (covariables A–D estandarizadas),
- un efecto espacial estructurado tipo **ICAR** usando la vecindad \(W\) (Queen),
- un efecto temporal tipo **Random Walk de orden 1 (RW1)** mensual,
y generar como producto final las **superficies posteriores** por celda–mes (media e intervalo creíble).

### ¿Qué hace esta celda?

1. **Carga los insumos generados en la CELDA 1:**
   - `HBM_panel_PM25.parquet` (y + X estandarizadas + índices)
   - `HBM_W_edges_queen.csv` (aristas i→j)
   - `HBM_meta.json` (lista de covariables y configuración)

2. **Construye la estructura espacial ICAR:**
   - usa las aristas \((i,j)\) para penalizar diferencias \((\phi_i - \phi_j)^2\),
   - impone centrado \(\sum_i \phi_i = 0\) para identificabilidad.

3. **Define el componente temporal RW1:**
   - un efecto \(\delta_t\) que evoluciona suavemente mes a mes,
   - centrado para identificabilidad.

4. **Ajusta el HBM (PM2.5):**
   - modelo normal para \(y_{i,t}\) (en log si así se definió),
   - \( \mu_{i,t} = \alpha + X_{i,t}\beta + \phi_i + \delta_t \)

5. **Genera el producto final (lo esperado):**
   - calcula la **media posterior** y el **intervalo creíble 95%** de la concentración (en escala original) para cada celda–mes,
   - guarda un CSV listo para mapas/QGIS y para escenarios posteriores.

### Salida esperada
- `CODIGO/HBM_OUT/HBM_PM25_base_pred_2020_2024.csv`
  con columnas:
  - `cell_id`, `fecha`, `mean_hbm`, `lo95_hbm`, `hi95_hbm`

In [3]:
import os
os.environ["PYTENSOR_FLAGS"] = "base_compiledir=C:\\pytensor_cache,optimizer_excluding=local_subtensor_merge"
# ✅ CELDA 2 — HBM base PM2.5 (ICAR espacial + RW1 temporal) + exportar superficies posteriores
import pymc as pm 
import pytensor
import numpy as np
import pandas as pd
import json
from pathlib import Path

# ---------- localizar OUT_DIR (misma lógica: ejecutas desde CODIGO en VS Code) ----------
cwd = Path.cwd()
if cwd.name.lower() == "codigo":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "CODIGO").exists()), cwd)

OUT_DIR = PROJECT_ROOT / "CODIGO" / "HBM_OUT"
panel_path = OUT_DIR / "HBM_panel_PM25.parquet"
edges_path = OUT_DIR / "HBM_W_edges_queen.csv"
meta_path  = OUT_DIR / "HBM_meta.json"

print("📌 OUT_DIR =", OUT_DIR)
for p in [panel_path, edges_path, meta_path]:
    print(" -", p, "| exists:", p.exists())
    if not p.exists():
        raise FileNotFoundError(f"No encuentro: {p}")

# ---------- dependencias bayesianas ----------
try:
    import pymc as pm
    import pytensor.tensor as at
    import arviz as az
except Exception as e:
    raise ImportError(
        "❌ Falta PyMC/ArviZ en tu entorno. Instala en tu .venv:\n"
        "   pip install pymc arviz pytensor\n"
        "y vuelve a correr esta celda.\n"
        f"Detalle: {e}"
    )

# ---------- cargar datos ----------
df = pd.read_parquet(panel_path)
edges = pd.read_csv(edges_path)
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

POLL = meta["POLL"]
USE_LOG = bool(meta["USE_LOG"])
X_cols = meta["X_cols"]

# ---------- matrices para el modelo ----------
y = df["y"].to_numpy().astype(float)
cell_idx = df["cell_idx"].to_numpy().astype(int)
time_idx = df["time_idx"].to_numpy().astype(int)
X = df[X_cols].to_numpy().astype(float)

n = int(df["cell_idx"].max()) + 1
T = int(df["time_idx"].max()) + 1
p = X.shape[1]

# edges: volverlos NO dirigidos (pares únicos i<j)
e_i = edges["i"].to_numpy().astype(int)
e_j = edges["j"].to_numpy().astype(int)
ii = np.minimum(e_i, e_j)
jj = np.maximum(e_i, e_j)
pairs = np.unique(np.stack([ii, jj], axis=1), axis=0)
ei = pairs[:, 0]
ej = pairs[:, 1]

print("\n===== RESUMEN HBM BASE =====")
print("POLL:", POLL)
print("N obs:", len(df), "| n_celdas:", n, "| n_meses:", T, "| p covariables:", p)
print("X_cols:", X_cols)
print("USE_LOG:", USE_LOG, "(y ya viene transformada desde CELDA 1)")
print("Edges (undirected):", len(pairs))

# ---------- modelo bayesiano ----------
with pm.Model() as model:
    # intercepto y betas
    alpha = pm.Normal("alpha", mu=0.0, sigma=2.0)
    beta  = pm.Normal("beta",  mu=0.0, sigma=1.0, shape=p)  # X ya está estandarizada

    # ruido observacional
    sigma_y = pm.Exponential("sigma_y", 1.0)

    # ---- espacial ICAR: penaliza (phi_i - phi_j)^2 y centra a suma cero ----
    tau_phi = pm.Exponential("tau_phi", 1.0)  # precisión/regularización espacial
    phi_raw = pm.Normal("phi_raw", 0.0, 1.0, shape=n)
    phi = pm.Deterministic("phi", phi_raw - at.mean(phi_raw))  # constraint sum-to-zero

    pm.Potential("icar_penalty", -0.5 * tau_phi * at.sum((phi[ei] - phi[ej]) ** 2))

    # ---- temporal RW1: suavidad mensual (random walk) y centrado ----
    sigma_t = pm.Exponential("sigma_t", 1.0)
    delta_raw = pm.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=T)
    delta = pm.Deterministic("delta", delta_raw - at.mean(delta_raw))

    # media latente
    mu = alpha + at.dot(X, beta) + phi[cell_idx] + delta[time_idx]

    # likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma_y, observed=y)

    # muestreo (config “segura”)
    idata = pm.sample(
        draws=1500,
        tune=1500,
        chains=4,
        target_accept=0.95,
        random_seed=42,
        progressbar=True
    )

# ---------- postproceso: superficies posteriores en escala original ----------
# (Esto exporta la *superficie latente* exp(mu), no la predicción con ruido)
post = idata.posterior

alpha_s = post["alpha"].values.reshape(-1)                 # (S,)
beta_s  = post["beta"].values.reshape(-1, p)               # (S,p)
phi_s   = post["phi"].values.reshape(-1, n)                # (S,n)
delta_s = post["delta"].values.reshape(-1, T)              # (S,T)

S = alpha_s.shape[0]
Nobs = len(y)

# mu por draw y observación: S x N
mu_s = alpha_s[:, None] + (beta_s @ X.T) + phi_s[:, cell_idx] + delta_s[:, time_idx]

# convertir a concentración (latente). Si y era log(C+eps), la inversa es exp(mu)-eps
eps = 1e-6
C_s = np.exp(mu_s) - eps

mean_hbm = np.mean(C_s, axis=0)
lo95_hbm = np.quantile(C_s, 0.025, axis=0)
hi95_hbm = np.quantile(C_s, 0.975, axis=0)

out = df[["cell_id", "fecha"]].copy()
out["mean_hbm"] = mean_hbm
out["lo95_hbm"] = lo95_hbm
out["hi95_hbm"] = hi95_hbm

out_path = OUT_DIR / "HBM_PM25_base_pred_2020_2024.csv"
out.to_csv(out_path, index=False)

# resumen corto de parámetros
sum_path = OUT_DIR / "HBM_PM25_base_summary.csv"
az.summary(idata, var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"]).to_csv(sum_path)

print("\n✅ Listo. Salidas:")
print(" -", out_path)
print(" -", sum_path)
print("\n📌 Siguiente esperado: usar este CSV para mapas/validación y luego repetir para NO2 y O3.")

📌 OUT_DIR = d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_panel_PM25.parquet | exists: True
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_W_edges_queen.csv | exists: True
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_meta.json | exists: True

===== RESUMEN HBM BASE =====
POLL: PM25
N obs: 15240 | n_celdas: 254 | n_meses: 60 | p covariables: 3
X_cols: ['diff_PM25_grid', 'grad_PM25_grid', 'adv_proxy_PM25_grid']
USE_LOG: True (y ya viene transformada desde CELDA 1)
Edges (undirected): 805


d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pymc\distributions\timeseries.py:291: UserWarning: Initial distribution not specified, defaulting to `Normal.dist(0, 100)`.You can specify an init_dist manually to suppress this warning.
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, beta, sigma_y, tau_phi, phi_raw, sigma_t, delta_raw]


Output()

Sampling 4 chains for 1_500 tune and 1_500 draw iterations (6_000 + 6_000 draws total) took 239 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



✅ Listo. Salidas:
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_PM25_base_pred_2020_2024.csv
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_PM25_base_summary.csv

📌 Siguiente esperado: usar este CSV para mapas/validación y luego repetir para NO2 y O3.


## CELDA 3 — Validación del HBM (PM2.5) + diagnóstico rápido (R-hat / ESS)

**Objetivo:** verificar que el producto del HBM para PM2.5 es coherente y cuantificar su desempeño con métricas.

### ¿Qué hace esta celda?
1) **Diagnóstico de convergencia (desde `HBM_PM25_base_summary.csv`):**
   - Identifica si existe algún parámetro con `r_hat > 1.01` o `ess_bulk < 100`.
   - Guarda un reporte en texto para anexos.

2) **Validación 1 (siempre disponible): HBM vs IDW en celda–mes**
   - Une `HBM_PM25_base_pred_2020_2024.csv` con `grid_3km_AD_mensual_2020_2024.csv` por (`cell_id`, `fecha`).
   - Calcula MAE, RMSE, sesgo, R y R² entre `mean_hbm` y `PM25_idw`.
   > Ojo: esto es chequeo de coherencia (HBM “regulariza” el campo), no es validación independiente.

3) **Validación 2 (opcional): HBM vs Observaciones (estaciones)**
   - Si encuentra un archivo de observaciones con coordenadas (`x`,`y`) y PM2.5 observado,
     asigna cada estación al **centroide de celda más cercano** y calcula métricas obs vs HBM.
   - Si no logra detectar esas columnas, lo reporta y continúa sin fallar.

### Salidas
- `HBM_PM25_diagnostics_report.txt`
- `HBM_PM25_metrics.csv`
- (opcional) `HBM_PM25_metrics_by_month.csv`

In [ ]:
# ✅ CELDA 3 — Validación HBM (PM2.5) + Diagnóstico

import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------
# 0) Rutas (auto)
# --------------------------
cwd = Path.cwd()
if cwd.name.lower() == "codigo":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "BASES DE DATOS").exists()), cwd)

DATA_DIR = PROJECT_ROOT / "BASES DE DATOS" / "PANEL AMBIENTAL INTEGRADO"
OUT_DIR  = PROJECT_ROOT / "CODIGO" / "HBM_OUT"

pred_path = OUT_DIR / "HBM_PM25_base_pred_2020_2024.csv"
sum_path  = OUT_DIR / "HBM_PM25_base_summary.csv"
ad_path   = DATA_DIR / "grid_3km_AD_mensual_2020_2024.csv"

for p in [pred_path, sum_path, ad_path]:
    if not p.exists():
        raise FileNotFoundError(f"No encuentro: {p}")

pred = pd.read_csv(pred_path)
summ = pd.read_csv(sum_path)
ad   = pd.read_csv(ad_path)

# --------------------------
# helpers
# --------------------------
def to_month_str(series):
    s = pd.to_datetime(series, errors="coerce")
    if s.isna().any():
        # si ya viene como YYYY-MM, intentamos normalizar por fallback
        s2 = pd.to_datetime(series.astype(str) + "-01", errors="coerce")
        if s2.isna().any():
            raise ValueError("No pude convertir 'fecha' a mes (datetime). Revisa el formato.")
        s = s2
    return s.dt.to_period("M").astype(str)

def metrics(obs, predv):
    obs = np.asarray(obs, dtype=float)
    predv = np.asarray(predv, dtype=float)
    m = ~np.isnan(obs) & ~np.isnan(predv)
    obs = obs[m]; predv = predv[m]
    out = {"n": int(len(obs))}
    if len(obs) == 0:
        out.update({"mae": np.nan, "rmse": np.nan, "bias": np.nan, "r": np.nan, "r2": np.nan})
        return out
    err = predv - obs
    out["mae"]  = float(np.mean(np.abs(err)))
    out["rmse"] = float(np.sqrt(np.mean(err**2)))
    out["bias"] = float(np.mean(err))
    if np.std(obs) == 0 or np.std(predv) == 0:
        out["r"] = np.nan
        out["r2"] = np.nan
    else:
        r = float(np.corrcoef(obs, predv)[0, 1])
        out["r"] = r
        out["r2"] = float(r**2)
    return out

# --------------------------
# 1) Diagnóstico rápido (R-hat / ESS)
# --------------------------
# nombre de la columna del parámetro
param_col = "Unnamed: 0" if "Unnamed: 0" in summ.columns else summ.columns[0]

rhat_col = "r_hat" if "r_hat" in summ.columns else None
ess_col  = "ess_bulk" if "ess_bulk" in summ.columns else None

diag_lines = []
diag_lines.append("HBM PM2.5 — Diagnóstico rápido\n")
diag_lines.append(f"Summary: {sum_path}\n")

if rhat_col and ess_col:
    bad_rhat = summ[summ[rhat_col] > 1.01]
    bad_ess  = summ[summ[ess_col] < 100]
    diag_lines.append(f"Parámetros con r_hat > 1.01: {len(bad_rhat)}")
    diag_lines.append(f"Parámetros con ess_bulk < 100: {len(bad_ess)}\n")

    if len(bad_rhat) > 0:
        diag_lines.append("Top r_hat (desc):")
        diag_lines.append(str(bad_rhat.sort_values(rhat_col, ascending=False).head(15)[[param_col, rhat_col, ess_col]]))
        diag_lines.append("")
    if len(bad_ess) > 0:
        diag_lines.append("Top ess_bulk (asc):")
        diag_lines.append(str(bad_ess.sort_values(ess_col, ascending=True).head(15)[[param_col, rhat_col, ess_col]]))
        diag_lines.append("")
else:
    diag_lines.append("No encontré columnas r_hat y/o ess_bulk en el summary.\n")

diag_path = OUT_DIR / "HBM_PM25_diagnostics_report.txt"
diag_path.write_text("\n".join(diag_lines), encoding="utf-8")

# --------------------------
# 2) Validación HBM vs IDW (celda–mes)
# --------------------------
# normalizar fechas a "YYYY-MM"
pred["fecha_m"] = to_month_str(pred["fecha"])
# ad puede tener 'fecha' o 'Fecha'
if "fecha" in ad.columns:
    ad["fecha_m"] = to_month_str(ad["fecha"])
elif "Fecha" in ad.columns:
    ad["fecha_m"] = to_month_str(ad["Fecha"])
else:
    raise ValueError("No encuentro columna 'fecha'/'Fecha' en grid_3km_AD_mensual_2020_2024.csv")

if "PM25_idw" not in ad.columns:
    raise ValueError("No encuentro PM25_idw en grid_3km_AD_mensual_2020_2024.csv")

ad_small = ad[["cell_id", "fecha_m", "PM25_idw"]].drop_duplicates()
pred_small = pred[["cell_id", "fecha_m", "mean_hbm", "lo95_hbm", "hi95_hbm"]].drop_duplicates()

m1 = pred_small.merge(ad_small, on=["cell_id", "fecha_m"], how="left")
met_idw = metrics(m1["PM25_idw"], m1["mean_hbm"])

# métricas por mes (útil para detectar meses raros)
by_month_rows = []
for mm, g in m1.groupby("fecha_m"):
    r = metrics(g["PM25_idw"], g["mean_hbm"])
    r["fecha_m"] = mm
    by_month_rows.append(r)
met_by_month = pd.DataFrame(by_month_rows).sort_values("fecha_m")

# --------------------------
# 3) Validación opcional vs estaciones (si se puede)
# --------------------------
# Buscamos un archivo típico de estaciones (si existe)
obs_candidates = [
    DATA_DIR / "panel_ambiental_mensual_2020_2024.csv",
    DATA_DIR / "panel_ambiental_mensual_2020_2024_con_tasas.csv",
]
obs_path = next((p for p in obs_candidates if p.exists()), None)

met_obs = None
note_obs = None

if obs_path is not None:
    obs = pd.read_csv(obs_path)

    # detectar fecha mensual
    if "fecha" in obs.columns:
        obs["fecha_m"] = to_month_str(obs["fecha"])
    elif "Fecha" in obs.columns:
        obs["fecha_m"] = to_month_str(obs["Fecha"])
    elif (("Año" in obs.columns or "Anio" in obs.columns) and ("Mes" in obs.columns or "mes" in obs.columns)):
        ycol = "Año" if "Año" in obs.columns else "Anio"
        mcol = "Mes" if "Mes" in obs.columns else "mes"
        obs["fecha_m"] = pd.to_datetime(dict(year=obs[ycol], month=obs[mcol], day=1)).dt.to_period("M").astype(str)
    else:
        obs["fecha_m"] = None

    # detectar coords (x,y) en metros
    xcol = None; ycol = None
    for cand in ["x", "X", "cx", "CX", "E", "este", "Este"]:
        if cand in obs.columns:
            xcol = cand; break
    for cand in ["y", "Y", "cy", "CY", "N", "norte", "Norte"]:
        if cand in obs.columns:
            ycol = cand; break

    # detectar PM2.5 observado (no IDW)
    pm_obs_col = None
    pm_candidates = [c for c in obs.columns if "PM25" in c.upper() and "IDW" not in c.upper()]
    if pm_candidates:
        pm_obs_col = pm_candidates[0]

    # si tenemos coords y pm obs y fecha:
    if (xcol is not None) and (ycol is not None) and (pm_obs_col is not None) and obs["fecha_m"].notna().any():
        # necesitamos centroids por celda para nearest neighbor.
        # los buscamos en ad (si tiene cx/cy). Si no, no podemos sin grid_clip.
        if ("cx" in ad.columns) and ("cy" in ad.columns):
            cent = ad[["cell_id", "cx", "cy"]].drop_duplicates()
            # KDTree / NearestNeighbors si existe; si no, brute por chunks
            pts = cent[["cx", "cy"]].to_numpy(dtype=float)
            cell_ids = cent["cell_id"].to_numpy()

            q = obs[[xcol, ycol]].to_numpy(dtype=float)

            cell_nn = np.empty(len(obs), dtype=cell_ids.dtype)

            # intentamos scipy
            used = None
            try:
                from scipy.spatial import cKDTree
                tree = cKDTree(pts)
                _, idx = tree.query(q, k=1)
                cell_nn = cell_ids[idx]
                used = "scipy.cKDTree"
            except Exception:
                # intentamos sklearn
                try:
                    from sklearn.neighbors import NearestNeighbors
                    nn = NearestNeighbors(n_neighbors=1, algorithm="auto")
                    nn.fit(pts)
                    _, idx = nn.kneighbors(q, n_neighbors=1)
                    cell_nn = cell_ids[idx[:,0]]
                    used = "sklearn.NearestNeighbors"
                except Exception:
                    # brute force por chunks
                    used = "brute_chunks"
                    chunk = 2000
                    for a in range(0, len(q), chunk):
                        qq = q[a:a+chunk]
                        d2 = ((qq[:,None,0]-pts[None,:,0])**2 + (qq[:,None,1]-pts[None,:,1])**2)
                        idx = np.argmin(d2, axis=1)
                        cell_nn[a:a+chunk] = cell_ids[idx]

            obs2 = obs.copy()
            obs2["cell_id"] = cell_nn
            # unir con predicciones HBM por cell_id y fecha_m
            join = obs2[["cell_id","fecha_m",pm_obs_col]].rename(columns={pm_obs_col:"PM25_obs"})
            m2 = pred_small.merge(join, on=["cell_id","fecha_m"], how="inner")
            met_obs = metrics(m2["PM25_obs"], m2["mean_hbm"])
            met_obs["method_cell_assign"] = used
            met_obs["obs_file"] = str(obs_path)
        else:
            note_obs = "No pude validar vs estaciones: grid_3km_AD no tiene cx/cy (centroides)."
    else:
        note_obs = "No pude validar vs estaciones: faltan columnas fecha mensual, x/y o PM25 observado en el archivo de estaciones."
else:
    note_obs = "No encontré archivo de observaciones por estación en PANEL AMBIENTAL INTEGRADO."

# --------------------------
# 4) Guardar métricas
# --------------------------
rows = [{"comparacion": "HBM_mean vs IDW (celda-mes)", **met_idw}]
if met_obs is not None:
    rows.append({"comparacion": "HBM_mean vs Observaciones (estaciones)", **met_obs})
elif note_obs is not None:
    rows.append({"comparacion": "HBM vs Observaciones (estaciones)", "note": note_obs})

met_df = pd.DataFrame(rows)
met_path = OUT_DIR / "HBM_PM25_metrics.csv"
met_df.to_csv(met_path, index=False)

met_by_month_path = OUT_DIR / "HBM_PM25_metrics_by_month.csv"
met_by_month.to_csv(met_by_month_path, index=False)

print("✅ Diagnóstico:", diag_path)
print("✅ Métricas:", met_path)
print("✅ Métricas por mes:", met_by_month_path)
print("\n=== MÉTRICAS (RESUMEN) ===")
print(met_df)

✅ Diagnóstico: d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_PM25_diagnostics_report.txt
✅ Métricas: d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_PM25_metrics.csv
✅ Métricas por mes: d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_OUT\HBM_PM25_metrics_by_month.csv

=== MÉTRICAS (RESUMEN) ===
                         comparacion        n       mae      rmse      bias  \
0        HBM_mean vs IDW (celda-mes)  15240.0  0.612766  0.818676 -0.010805   
1  HBM vs Observaciones (estaciones)      NaN       NaN       NaN       NaN   

          r        r2                                               note  
0  0.988897  0.977917                                                NaN  
1       NaN       NaN  No pude validar vs estaciones: faltan columnas...  


: 


## 1) Preparamos el panel para el HBM (CELDA 1)

* Tomamos el dataset final de la grilla: **`grid_3km_AD_mensual_2020_2024.csv`** (15.240 filas = 254 celdas × 60 meses).
* Definimos la **respuesta** del modelo para PM2.5:

  * (y_{i,t} = \log(PM25_idw + \epsilon)) (por eso `USE_LOG=True`).
* Seleccionamos covariables disponibles para PM2.5 (A–D):

  * `diff_PM25_grid`, `grad_PM25_grid`, `adv_proxy_PM25_grid`
* Creamos índices internos:

  * `cell_idx` para el efecto espacial,
  * `time_idx` para el efecto temporal.
* Convertimos la vecindad **Queen** del archivo **`W_grid_3km_queen.csv`** a aristas con índices (`i`, `j`).

✅ Productos guardados:

* `HBM_panel_PM25.parquet`
* `HBM_W_edges_queen.csv`
* `HBM_meta.json`

---

## 2) Ajustamos el HBM base para PM2.5 (CELDA 2)

* Ajustamos un **modelo bayesiano jerárquico espacio–temporal** en soporte celda–mes con:

  * efectos fijos: (\alpha + X_{i,t}\beta)
  * efecto espacial **ICAR** usando tu (W) (Queen)
  * efecto temporal **RW1** mensual
* Corrimos MCMC (NUTS) y verificamos que el ajuste **sí terminó**.

✅ Productos finales del HBM (lo más importante):

* `HBM_PM25_base_pred_2020_2024.csv`
  (superficie final por celda–mes: `mean_hbm`, `lo95_hbm`, `hi95_hbm`)
* `HBM_PM25_base_summary.csv`
  (resumen de parámetros globales con `r_hat` y `ess_bulk`)

Y comprobaste en el summary que:

* `r_hat ≈ 1.0`
* `ess_bulk` alto
  👉 Convergencia muy buena para los parámetros globales.

---

## 3) Validamos y diagnosticamos el resultado (CELDA 3)

* Generamos un reporte de diagnóstico:

  * `HBM_PM25_diagnostics_report.txt`
* Calculamos métricas de coherencia **HBM vs IDW** en celda–mes:

  * n=15240, MAE≈0.613, RMSE≈0.819, sesgo≈-0.011, r≈0.989, r²≈0.978
* Intentamos validación **HBM vs observaciones (estaciones)**, pero quedó en NaN porque el archivo detectado no tenía columnas suficientes para hacer el mapeo.

✅ Productos:

* `HBM_PM25_metrics.csv`
* `HBM_PM25_metrics_by_month.csv`

---

## Estado actual (dónde vamos)

✅ Ya tenemos el **HBM final para PM2.5** con superficies + incertidumbre y validación básica de coherencia.
🔜 Lo siguiente es repetir el mismo pipeline para **NO2** y **O3**